# Minh hoa: duong hoi quy & du doan thay doi qua tung epoch

Xem Gradient Descent **xoay dan duong thang** ve dung vi tri, va cac **diem du doan** (tren duong) tien dan ve diem that → phan du (residual) co ngan lai.

- Dung **1 dac trung** de ve duoc tren mat phang x–y.
- Class luu lai `(w, b)` **moi epoch** → tua lai thanh anh tung / animation.

> Ly thuyet: `linear-regression.md`, `gradient-descent.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

plt.rcParams["figure.figsize"] = (7, 5)

## 1. Class hoi quy co LUU LAI tham so moi epoch

Giong class truoc, them `self.params_history` luu `(w, b, loss)` sau moi vong lap de tua lai.

In [ ]:
class LinearRegression:
    """Hoi quy tuyen tinh + Gradient Descent (co learning rate decay).
    Luu lai (w, b, loss) moi epoch de minh hoa qua trinh hoc."""

    def __init__(self, lr=0.01, n_iters=200, decay=0.0):
        self.lr = lr
        self.n_iters = n_iters
        self.decay = decay
        self.w = None
        self.b = None
        self.params_history = []   # [(w_copy, b, loss), ...] moi epoch

    def predict(self, X):
        return X @ self.w + self.b

    def compute_loss(self, y, y_pred):
        return np.mean((y_pred - y) ** 2)

    def compute_gradients(self, X, y, y_pred):
        n = X.shape[0]
        error = y_pred - y
        dw = (2 / n) * (X.T @ error)
        db = (2 / n) * np.sum(error)
        return dw, db

    def _current_lr(self, t):
        return self.lr / (1.0 + self.decay * t)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        self.params_history = []

        for epoch in range(self.n_iters):
            y_pred = self.predict(X)
            loss = self.compute_loss(y, y_pred)
            # luu tham so HIEN TAI (truoc khi cap nhat) de ve
            self.params_history.append((self.w.copy(), self.b, loss))

            dw, db = self.compute_gradients(X, y, y_pred)
            lr_t = self._current_lr(epoch)
            self.w -= lr_t * dw
            self.b -= lr_t * db
        return self

## 2. Du lieu 1 dac trung & huan luyen

$y = 3x + 5 + \text{nhieu}$. Bat dau $w=0, b=0$ (duong nam ngang) → xem no xoay len.

In [ ]:
rng = np.random.default_rng(42)
n = 60
x = rng.uniform(0, 10, size=n)
X = x.reshape(-1, 1)               # shape (n, 1) cho 1 dac trung
true_w, true_b = 3.0, 5.0
y = true_w * x + true_b + rng.normal(0, 2.0, size=n)

model = LinearRegression(lr=0.02, n_iters=400, decay=0.0).fit(X, y)
print("So epoch luu lai:", len(model.params_history))
w_fin, b_fin, loss_fin = model.params_history[-1]
print(f"Cuoi cung: w={w_fin[0]:.3f} (that 3), b={b_fin:.3f} (that 5), loss={loss_fin:.3f}")

## 3. Anh tung tinh: nhieu duong chong len (nhat → dam theo epoch)

Duong **nhat** = epoch dau (con nam ngang), duong **dam** = epoch sau (da khop). Thanh mau ben phai = so epoch.

In [ ]:
xs = np.linspace(x.min(), x.max(), 100)
frames = np.unique(np.linspace(0, len(model.params_history) - 1, 25).astype(int))

fig, ax = plt.subplots()
ax.scatter(x, y, s=25, color="#333", alpha=0.7, label="Du lieu that", zorder=3)
cmap = plt.cm.viridis
for k, ep in enumerate(frames):
    w_e, b_e, _ = model.params_history[ep]
    ax.plot(xs, w_e[0] * xs + b_e, color=cmap(k / (len(frames) - 1)), lw=1.5, alpha=0.8)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, len(model.params_history)))
fig.colorbar(sm, ax=ax, label="Epoch")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Duong hoi quy xoay dan ve dung vi tri qua cac epoch")
ax.legend(loc="upper left")
plt.show()

## 4. Animation: duong + diem du doan + phan du chuyen dong theo epoch

- Cham den = du lieu **that** (co dinh).
- Duong cam = duong hoi quy hien tai.
- Cham cam = **du doan** $\hat{y}$ (nam tren duong) — tien dan ve diem that.
- Doan xam = **phan du** (residual) — ngan lai khi hoc tot.

> Chay o che do JS (`to_jshtml`) → khong can ffmpeg, bam Play ngay trong notebook.

In [ ]:
anim_frames = np.unique(np.linspace(0, len(model.params_history) - 1, 80).astype(int))

fig, ax = plt.subplots()
ax.scatter(x, y, s=25, color="#333", alpha=0.7, zorder=3)        # du lieu that (co dinh)
(line,) = ax.plot([], [], color="tab:orange", lw=2.5, zorder=4)  # duong hoi quy
(pred_pts,) = ax.plot([], [], "o", color="tab:orange", ms=4, zorder=5)  # diem du doan
resid_lines = [ax.plot([], [], color="gray", lw=0.8, alpha=0.6, zorder=2)[0] for _ in range(n)]
txt = ax.text(0.03, 0.92, "", transform=ax.transAxes, fontsize=11,
              bbox=dict(boxstyle="round", fc="white", alpha=0.8))

ax.set_xlim(x.min() - 0.5, x.max() + 0.5)
ax.set_ylim(y.min() - 3, y.max() + 3)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Hoi quy tuyen tinh hoc qua tung epoch")

def update(ep):
    w_e, b_e, loss_e = model.params_history[ep]
    y_line = w_e[0] * xs + b_e
    line.set_data(xs, y_line)
    y_hat = w_e[0] * x + b_e
    pred_pts.set_data(x, y_hat)
    for i in range(n):
        resid_lines[i].set_data([x[i], x[i]], [y[i], y_hat[i]])
    txt.set_text(f"Epoch {ep:3d}\nw={w_e[0]:.2f}  b={b_e:.2f}\nloss={loss_e:.2f}")
    return [line, pred_pts, txt, *resid_lines]

anim = animation.FuncAnimation(fig, update, frames=anim_frames, interval=120, blit=True)
plt.close(fig)   # tranh hien khung tinh thua
HTML(anim.to_jshtml())

## ✅ Rut ra

- Bat dau $w=b=0$ → duong **nam ngang**, phan du **dai**.
- Moi epoch: gradient keo $w, b$ → duong **xoay + tinh tien** dan ve dung vi tri, phan du **ngan lai**, loss **giam**.
- Hoi tu: duong gan nhu khong doi nua (gradient ~ 0).

> Doi sang du lieu nhieu hon / lr khac / them `decay` de xem toc do hoc thay doi the nao.